<a href="https://colab.research.google.com/github/Dishitha09/csda-al/blob/main/notebooks/CSDA_AL_ISIC2019_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CSDA-AL: Class-Sensitive Deep Active Learning on ISIC 2019
### Master Testing & Controlled Validation Suite (A0 to A14)

This notebook provides the complete experimental and testing pipeline for **CSDA-AL** on the **ISIC 2019 dermoscopy dataset** (25,331 images, 8 classes, 53.9:1 class imbalance ratio) under strictly identical canonical splits and budget parity.

**Included Testing Modules:**
- **Step 1:** Instant Clean Reset & Setup (depth-1 clone, 10-second setup)
- **Step 2:** Technical Audit & Root-Cause Diagnosis (`--mode diagnose`)
- **Step 3:** Isolated A9 Experiment (Cold-Start + Class-Weighted Synergy across 3 seeds)
- **Step 4:** A14 Controlled Head-to-Head Validation (A1 vs A11 from Seed-Matched Pools)
- **Step 5:** Next-Stage Experiments A9 to A13 (`--mode next_stage`)
- **Step 6:** Baseline vs Improved CSDA-AL (`--mode improve`)
- **Step 7:** Display Master Benchmark Table & A14 Head-to-Head Table
- **Step 8:** Display Publication Figures Inline

## Step 1: Clean Reset & Fast Setup
Resets working directory to `/content`, wipes any stale folders, shallow-clones the latest repo, and links `src/`.

In [ ]:
import os
import sys
import torch

print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device:", torch.cuda.get_device_name(0))

# 1. Reset working directory to /content root
%cd /content

# 2. Force remove any stale or nested csda-al directory
!rm -rf /content/csda-al

# 3. Clone fresh repository (shallow clone for maximum speed)
!git clone --depth 1 https://github.com/Dishitha09/csda-al.git /content/csda-al
%cd /content/csda-al

# 4. Instant install without slow dependency checks
!pip install -q --no-deps -e .

# 5. Add src to python module path
if "/content/csda-al/src" not in sys.path:
    sys.path.insert(0, "/content/csda-al/src")

print("
✓ Repository ready at:", os.getcwd())
print("✓ Available scripts in repo:", sorted([f for f in os.listdir("scripts") if f.endswith(".py")]))


## Step 2: Technical Audit & Root-Cause Diagnosis (`--mode diagnose`)
Audits dataset class distribution, reveals the **Argmax Quota Trap**, **Loss Weight Singularity**, and **Cold-Start Density Bias**.

In [ ]:
%cd /content/csda-al
!python run_isic_pipeline.py --mode diagnose


## Step 3: Run Isolated A9 Experiment (Cold-Start + Class-Weighted Synergy)
Directly evaluates the combination of the two strongest baseline approaches (Calibrated Cold-Start + Class-Weighted Training) vs A0, A1, A2 under the verified fixed split across seeds `42, 43, 44`.

In [ ]:
%cd /content/csda-al
!python scripts/run_a9_isolated.py


## Step 4: Run A14 Controlled Head-to-Head Validation
Fair, seed-matched head-to-head comparison between **A1 (Class-Weighted Random)** and **A11 (Uncertainty + Revised Cosine Diversity)** starting from **identical initial labeled pools** across seeds `42, 43, 44`.

In [ ]:
%cd /content/csda-al
!python scripts/run_a14_controlled_validation.py


## Step 5: Execute Next-Stage Controlled Experiments A9 to A13 (`--mode next_stage`)
Runs the full suite of next-stage ablations:
- **A9**: Class-Weighted Training + Calibrated Cold-Start
- **A10**: Class-Weighted Training + Uncertainty Acquisition
- **A11**: Class-Weighted + Uncertainty + Revised Cosine Diversity
- **A12**: Validation-Tuned CSDA-AL Weights Grid Search
- **A13**: Final Integrated CSDA-AL on Locked Test Set

In [ ]:
%cd /content/csda-al
!python run_isic_pipeline.py --mode next_stage --seeds 42 43 44


## Step 6: Run Baseline vs Improved CSDA-AL Across 3 Seeds (`--mode improve`)
Executes Random vs Improved CSDA-AL across seeds `42, 43, 44` with bounded class-balanced loss and soft-target deficit matching.

In [ ]:
%cd /content/csda-al
!python run_isic_pipeline.py --mode improve --seeds 42 43 44


## Step 7: Display Comparison Tables & Summaries
Displays the master benchmark matrix (A0 to A13) and the A14 seed-matched head-to-head results.

In [ ]:
import os
import pandas as pd
from IPython.display import display

# Master benchmark A0-A13
p_master = "outputs/isic2019/metrics/next_stage_benchmark_comparison.csv"
if os.path.exists(p_master):
    print("=== Master Benchmark Matrix (A0-A13) ===")
    df_master = pd.read_csv(p_master)
    pd.set_option("display.max_columns", None)
    pd.set_option("display.width", 1000)
    display(df_master)

# A14 Head-to-Head
p_a14 = "outputs/isic2019/experiments/A14/a14_head_to_head_summary.csv"
if os.path.exists(p_a14):
    print("
=== A14 Seed-Matched Head-to-Head (A1 vs A11) ===")
    df_a14 = pd.read_csv(p_a14)
    display(df_a14)


## Step 8: Display Publication Figures Inline
Visualizes confusion matrices, learning curves, and per-class diagnostic recovery.

In [ ]:
import os
from IPython.display import Image, display

fig_pairs = [
    ("Figure 10: Confusion Matrix Comparison (Baseline vs Proposed CSDA-AL C4)", "results/figures/fig10_confusion_matrix_comparison.png"),
    ("Figure 6: Macro-F1 Score vs Annotation Budget", "results/figures/fig6_macro_f1_vs_budget.png"),
    ("Figure 11: Per-Class F1-Score (Eliminating Minority Diagnostic Collapse)", "results/figures/fig11_per_class_performance.png")
]

for title, path in fig_pairs:
    if os.path.exists(path):
        print(f"
{title}")
        display(Image(path, width=800))
